# Stratified study — turnover and buying pressure, by security size

New file. `turnover_study.py` and `buy_pressure.py` are untouched; `stratified_study.py` is
self-contained.

Everything runs **separately inside each `security_size` bucket**, plus a pooled run for
comparison. A pooled result is dominated by whichever bucket has the most names, and it can
hide a sign that flips between them. Quintiles are formed **inside** the bucket, so a large
cap is never ranked against a micro cap.

## Timing — the one thing to get right

Write $\mathcal I_t$ for what is knowable once quarter $t$ has closed.

| column | window it describes | measurable | may be a feature? |
|---|---|---|---|
| `security_size`, `fund_size` | a classification at q | $\mathcal I_q$ | key, not a feature |
| `active_weight` | a state at q | $\mathcal I_q$ | **yes, as is** |
| `security_*_fund_turnover` | this quarter's weight vs last quarter's, q−1 → q | $\mathcal I_q$ | **yes, as is** |
| `chg_pct` | q → q+1 | $\mathcal I_{q+1}$ | target, or lagged |
| `weight` → dw | q → q+1 | $\mathcal I_{q+1}$ | target, or lagged |

The three `*_fund_turnover` columns look **backwards**, unlike `chg_pct`, so they do **not**
need lagging. If your build actually defines them forwards, set
`assume_fund_turnover_backward = False` and they get lagged one quarter.

## The two new pressure ratios

$$R_w(s,q)=\frac{\sum_f \max(0,\Delta w_{f})}{\sum_f |\Delta w_{f}|}
\qquad
R_\$(s,q)=\frac{\sum_f \max(0,d_{f})}{\sum_f |d_{f}|},\quad d_f=\texttt{chg\_pct}_f\cdot\texttt{position\_value}_f$$

Both live in $[0,1]$ and are normalised by **total** activity. That separates two things every
other measure conflates: *how much* trading happened, and *which way* it went. 0.5 means
buying and selling balanced; 1 means every dollar of reallocation was a purchase.

Compare them against `net_flow`, which is a net number and so scores a heavily traded name
and a quiet one on the same scale even when their buy/sell balance is identical.


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import stratified_study as S
S.check_version()
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)


## 1. Configuration

In [ ]:
CFG = S.Config(
    holdings_path = "manager_holdings/master_batches_return_filtered/master_all_funds_add_filter_ivy_rank_active_rank.parquet",
    inv_type_codes = (401,),

    # set False if security_*_fund_turnover describes q -> q+1 rather than q-1 -> q
    assume_fund_turnover_backward = True,

    strata = (0, 1, 2),          # 0 small, 1 mid, 2 large
    min_stratum_rows = 2000,

    min_quarters = 8,
    window_q = 28, test_q = 8, step = 8,

    model = "hgb",               # "hgb" or "linear"
    train_target_transform = "rank",   # rank matches how the result is graded
)
CFG


## 2. Build the panel

Read these lines before anything else:

- **`[scale] chg_pct` / `[scale] weight`** — percent or fraction. Wrong here and every
  pressure measure below is wrong by 100x.
- **`[turnover] using:`** — which volume convention was picked.
- **`rows per security_size`** — a bucket far smaller than the others will have noisy folds.
- **`pressure measures`** coverage — a measure well below 100% is thin, and its stratified
  results will be the first to fall apart.

In [ ]:
panel = S.build_panel(CFG)
print("\nfeatures :", S.feature_list(panel))
print("pressure :", S.pressure_list(panel))


In [ ]:
# the two new ratios: bounded, centred near 0.5, and NOT redundant with net_flow
cols = [c for c in ["buy_frac","dollar_buy_frac","flow_pct_cap","weight_chg",
                    "active_weight_chg","buy_weight_ratio","buy_dollar_ratio"]
        if c in panel.columns]
display(panel[cols].describe().T.round(4))
print("rank correlation between the measures — far below 1 means they ask different questions:")
display(panel[cols].corr(method="spearman").round(3))


## 3. Turnover and returns, by size

`Q5_Q1_per_q` is the quintile spread in return per quarter and is comparable across targets;
`rank_IC` is against that row's own target.

In [ ]:
TAB_RET = S.run_stratified(panel, CFG, targets=["turnover_next", "ret_next"])
display(S.summary(TAB_RET))


In [ ]:
print("model:ALL vs the best single characteristic (compared on |IC|, since a raw sort's")
print("sign is arbitrary — flip it and you have the same information):")
display(S.beats_naive(TAB_RET))


## 4. Buying pressure, by size — all seven definitions

This is the run that answers the manager's question. Each target is one definition of
"buying pressure"; each stratum is one size bucket.

In [ ]:
PRESSURE_TARGETS = [S.PRESSURE[k] for k in S.pressure_list(panel)]
print(PRESSURE_TARGETS)
TAB_BP = S.run_stratified(panel, CFG, targets=PRESSURE_TARGETS)
display(S.summary(TAB_BP))


In [ ]:
display(S.beats_naive(TAB_BP))
print("edge > 0 means the feature set adds something beyond a one-characteristic sort.")
print("Buying pressure is strongly autocorrelated, so a large rank_IC with edge ~ 0 means")
print("the model learned only that it repeats.")


### Is the ratio a better target than the net measures?

`net_flow` and `weight_change` are **net** quantities, so a heavily traded name and a quiet
one land far apart even when their buy/sell balance is identical. The ratios strip the volume
out. If the ratios are more predictable, that difference in scale was noise; if less, the
volume itself carried information.

In [ ]:
cmp = (TAB_BP[TAB_BP.model == "model:ALL"]
       .pivot_table(index="target", columns="stratum", values="rank_IC").round(4))
order = [t for t in ["flow_pct_cap","buy_dollar_ratio","weight_chg","active_weight_chg",
                     "buy_weight_ratio","buy_frac","dollar_buy_frac"] if t in cmp.index]
display(cmp.loc[order])


## 5. Does predicted buying pressure carry alpha, by size?

Every row's `Q5_Q1_per_q` is a **return** spread, whatever the target was — so a
buying-pressure model and a return model sit on one scale and can be read off the same
column.

In [ ]:
alpha = (TAB_BP[TAB_BP.model == "model:ALL"]
         .pivot_table(index="target", columns="stratum",
                      values=["Q5_Q1_per_q","spread_t"]).round(3))
display(alpha)
print("Watch for a sign that FLIPS between small and large. That is the single most useful")
print("thing stratification can show, and it is invisible in the pooled row.")


## 6. Reading it

- **Sign flips across strata** — the reason to stratify at all. A pooled spread of zero can
  be a large positive in small caps cancelling a large negative in large caps.
- **`edge` near zero everywhere** — the features add nothing beyond persistence. Report that;
  do not tune until something appears.
- **A stratum with few `n_quarters`** — its folds were thin. Treat its t-stats as indicative.
- **Ratios vs net measures** — if the ratios predict better, the scale in `net_flow` was
  noise. Say which you are reporting and why.
- **Returns are gross** — no costs, and a turnover or pressure sort concentrates in exactly
  the names where trading costs differ most, so costs will not be neutral across quintiles.


In [ ]:
os.makedirs("outputs_stratified", exist_ok=True)
TAB_RET.to_csv("outputs_stratified/stratified_returns.csv", index=False)
TAB_BP.to_csv("outputs_stratified/stratified_buy_pressure.csv", index=False)
panel.to_parquet("outputs_stratified/stratified_panel.parquet", index=False)
print("saved to outputs_stratified/")
